In [2]:
import os
import pandas as pd

# Define the input and output directories
directory = '/home/ahmedmas/Projects/Data_imputation/Modified_Data'
output_directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to calculate missing values and percentages
def calculate_missing_values(df):
    missing_values = df.isnull().sum()
    total_values = len(df)
    missing_percent = (missing_values / total_values) * 100
    return missing_values, missing_percent

# Traverse the directory and process each CSV file
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        study_site = extract_study_site(filename)
        filepath = os.path.join(directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Calculate missing values for each column
        missing_values, missing_percent = calculate_missing_values(df)

        # Delete columns with more than 50% missing values
        columns_to_delete = missing_percent[missing_percent > 50].index
        df.drop(columns=columns_to_delete, inplace=True)

        # Set the datetime column as the index
        df.set_index('datetime', inplace=True)

        # Interpolate missing values for all columns except PM2.5
        columns_to_interpolate = df.columns.difference(['PM2.5'])
        df[columns_to_interpolate] = df[columns_to_interpolate].interpolate(method='time')

        # Reset the index to save the DataFrame
        df.reset_index(inplace=True)

        # Save the modified DataFrame to the output directory with the same filename
        output_filepath = os.path.join(output_directory, filename)
        df.to_csv(output_filepath, index=False)
        print(f"Processed and saved file: {output_filepath}")

print("Processing complete.")


Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_BATHURST_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_WAGGA_2018-12-31_2024-07-16_data.csv
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Processed_Data/AQMS_WOLLONGONG_2018-12-31_2024-07-1

In [ ]:
import os
import pandas as pd
import numpy as np

# Define the input and output directories
input_directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'
output_directory = '/home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to calculate missing percentage
def calculate_missing_percentage(df, column):
    total_values = len(df)
    missing_values = df[column].isnull().sum()
    missing_percentage = (missing_values / total_values) * 100
    return missing_percentage, missing_values, total_values

# Traverse the directory and process each CSV file
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(input_directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Calculate initial missing percentage of PM2.5
        initial_missing_percentage, initial_missing_count, total_values = calculate_missing_percentage(df, 'PM2.5')

        # Determine the number of additional values to remove to achieve 10% missing
        target_missing_percentage = 10
        target_missing_count = int(target_missing_percentage * total_values / 100)
        additional_missing_count = target_missing_count - initial_missing_count

        if additional_missing_count > 0:
            # Randomly select indices to set as NaN
            non_missing_indices = df[df['PM2.5'].notnull()].index
            if len(non_missing_indices) < additional_missing_count:
                print(f"Not enough non-missing values in {filename} to achieve 10% missing")
                continue
            additional_missing_indices = np.random.choice(non_missing_indices, additional_missing_count, replace=False)
            df.loc[additional_missing_indices, 'PM2.5'] = np.nan

            # Save the modified DataFrame to the output directory with the same filename
            output_filepath = os.path.join(output_directory, filename)
            df.to_csv(output_filepath, index=False)
            print(f"Processed and saved file: {output_filepath} with {target_missing_percentage}% missing PM2.5 values")

print("Processing complete.")


Making 10% removal for PM2.5

Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_BATHURST_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modifie

In [5]:
import os
import pandas as pd
import numpy as np

# Define the input and output directories
input_directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'
output_directory = '/home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to calculate missing percentage
def calculate_missing_percentage(df, column):
    total_values = len(df)
    missing_values = df[column].isnull().sum()
    missing_percentage = (missing_values / total_values) * 100
    return missing_percentage, missing_values, total_values

# Traverse the directory and process each CSV file
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        study_site = extract_study_site(filename)
        filepath = os.path.join(input_directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Calculate initial missing percentage of PM2.5
        initial_missing_percentage, initial_missing_count, total_values = calculate_missing_percentage(df, 'PM2.5')

        # Create a column to mark originally missing data
        df['missing_info'] = np.where(df['PM2.5'].isnull(), 'OM', '')

        # Determine the number of additional values to remove to achieve 10% missing
        target_missing_percentage = 10
        target_missing_count = int(target_missing_percentage * total_values / 100)
        additional_missing_count = target_missing_count - initial_missing_count

        # Create a column to store removed values
        df['removed_values'] = np.nan

        if additional_missing_count > 0:
            # Randomly select indices to set as NaN
            non_missing_indices = df[df['PM2.5'].notnull()].index
            if len(non_missing_indices) < additional_missing_count:
                print(f"Not enough non-missing values in {filename} to achieve 10% missing")
                continue
            additional_missing_indices = np.random.choice(non_missing_indices, additional_missing_count, replace=False)
            df.loc[additional_missing_indices, 'removed_values'] = df.loc[additional_missing_indices, 'PM2.5']
            df.loc[additional_missing_indices, 'PM2.5'] = np.nan

            # Mark the artificially missing data
            df.loc[additional_missing_indices, 'missing_info'] = 'AM'

        # Add columns for hour, day, and month
        df['Hour'] = df['datetime'].dt.hour
        df['Day'] = df['datetime'].dt.day
        df['Month'] = df['datetime'].dt.month

        # Save the modified DataFrame to the output directory with the same filename
        output_filepath = os.path.join(output_directory, filename)
        df.to_csv(output_filepath, index=False)
        print(f"Processed and saved file: {output_filepath} with {target_missing_percentage}% missing PM2.5 values")

print("Processing complete.")


Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_BATHURST_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv with 10% missing PM2.5 values
Processed and saved file: /home/ahmedmas/Projects/Data_imputation/Modifie